# Sampling Distributions And Bootstrap Intuition

**Official MA1001B Alignment:** *4.1 transformations; 4.2 functions of random variables; 4.3 sampling distribution of the mean.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Distinguish between the distribution of individual observations and the sampling distribution of a sample statistic.
- Calculate the theoretical standard error of the mean (`s / sqrt(n)`) from a single observed sample.
- Implement non-parametric Bootstrap resampling using Pandas `.sample(replace=True)` to simulate sampling variability.
- Construct and interpret 95% Bootstrap confidence intervals to quantify estimation precision.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model sample-to-sample variability (standard error) to understand how much a sample mean fluctuates around the true population parameter.
- **2. Computational Link (How Python represents it):** We use list comprehensions and Pandas resampling with replacement (`replace=True`) to computationally generate bootstrap distributions.
- **3. Decision Link (How it guides action):** Quantifying sampling variability prevents stakeholders from overreacting to minor sample-to-sample fluctuations.


## Decision Scenario

> **The Problem:** A public-policy analyst has one sample and one average score. The stakeholder wants to know how much that average would vary if a different sample had been collected.


## Conceptual Explanation

A statistic is a number computed from a sample, but it varies from sample to sample. A sampling distribution describes that variation. The bootstrap approximates sampling variation by repeatedly resampling from the observed data.


## Mathematical Anchor

For independent observations with standard deviation sigma, the standard error of the mean is approximately sigma / sqrt(n). In practice sigma is often estimated by the sample standard deviation.


## Data And Workflow Notes

Uses simulated score data to make repeated sampling visible.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Population Setup & Single Sample Extraction

We generate a large reference population of 50,000 policy scores and extract a single operational sample of n=120 observations to compute initial sample statistics.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Create reference population of 50,000 scores and draw a single sample of n=120
population = pd.Series(rng.normal(loc=6.2, scale=1.1, size=50_000), name="policy_score").clip(0, 10)
sample = population.sample(n=120, random_state=1001)

# Compute single sample statistics and theoretical standard error of the mean
pd.Series({
    "sample_mean_xbar": sample.mean(),
    "sample_sd_s": sample.std(ddof=1),
    "theoretical_standard_error_(s/sqrt(n))": sample.std(ddof=1) / np.sqrt(len(sample)),
}).round(4)


### Step 2: True Sampling Distribution (Repeated Sampling)

To demonstrate what a sampling distribution is, we draw 1,000 independent samples directly from the known population and inspect the distribution of their sample means.


In [ ]:
# Draw 1,000 independent samples from population to observe true sampling variability
repeated_means = []
for seed in range(1000):
    repeated_sample = population.sample(n=120, random_state=seed)
    repeated_means.append(repeated_sample.mean())

# Summarize the true sampling distribution of the mean
pd.Series(repeated_means, name="true_sampling_distribution_means").describe().round(4)


### Step 3: Bootstrap Resampling from a Single Sample

In real data science, we only have one sample. We use Bootstrap resampling (drawing n=120 with replacement from our single sample 1,000 times) to approximate the sampling distribution.


In [ ]:
# Generate 1,000 bootstrap resamples from our single observed sample
bootstrap_means = [
    sample.sample(n=len(sample), replace=True, random_state=seed).mean()
    for seed in range(1000)
]

# Calculate 95% Bootstrap percentile confidence interval
pd.Series(bootstrap_means, name="bootstrap_means").quantile([0.025, 0.50, 0.975]).round(4)


### Step 4: Visualizing Bootstrap vs. Sample Mean

We plot the histogram of the 1,000 Bootstrap sample means, marking the original sample mean and the 95% confidence interval bounds.


In [ ]:
# Plot Bootstrap sampling distribution with confidence bounds
ax = sns.histplot(bootstrap_means, kde=True, color="royalblue", bins=30)
ax.axvline(sample.mean(), color="red", linestyle="--", linewidth=2, label=f"Sample Mean: {sample.mean():.2f}")
ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
ax.axvline(ci_low, color="black", linestyle=":", linewidth=1.5, label=f"95% CI: [{ci_low:.2f}, {ci_high:.2f}]")
ax.axvline(ci_high, color="black", linestyle=":", linewidth=1.5)
ax.set_title("Bootstrap Sampling Distribution of the Mean (n=120)", fontsize=14, pad=10)
ax.set_xlabel("Bootstrap Sample Mean", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.legend()
plt.show()


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Explain in your own words the difference between the distribution of individual scores and the sampling distribution of sample means.

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Thinking that Bootstrap resampling creates new independent data from nothing (it only approximates sampling variability).
- **Warning:** Confusing the standard deviation of individual observations (`s`) with the standard error of a sample statistic (`s / sqrt(n)`).
- **Warning:** Assuming Bootstrap intervals are valid when the initial sample was collected with severe selection bias.


## Independent Practice

> [!TIP]
> **Your Task:**
> Repeat the Bootstrap resampling procedure using simulated sample sizes of `n=40` and `n=300`. Compare the width of the resulting 95% confidence intervals.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** Why does statistical uncertainty (standard error) shrink as the sample size `n` increases?

*Write your brief conceptual reflection below:*
